In [1]:
!pip install pulp
from pulp import *
import numpy as np
import random
import pprint
from collections import defaultdict

import pandas as pd
import json
import re
import os
from datetime import datetime, timedelta
import time


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.4/16.4 MB 86.1 MB/s eta 0:00:00


In [5]:
### Funciones auxiliares
#Función para calcular la penalización por el tiempo restante al vencimiento de una órden
def fec_o(t,M =100000000000,alpha = 0.5, w = 4):
    if t <= 5: #(en minutos)
        return M
    return float(alpha * np.exp(-(t-5.0)/w))


def imprimir_listas_no_nulas(estructura):
    """
    Imprime las listas no vacías en una estructura de listas anidadas
    """
    for i, fila in enumerate(estructura):
        listas_no_vacias = []
        for j, lista in enumerate(fila):
            if lista:  # Si la lista no está vacía
                listas_no_vacias.append(f"fila[{i}][{j}]: {lista}")

        if listas_no_vacias:
            print(f"Fila {i}:")
            for item in listas_no_vacias:
                print(f"  {item}")


def generar_ventana(tiempo_desde, tiempo_hasta):
    # BACKLOG - Filtrar órdenes que SÍ están en la ventana temporal
    # Archivos recibidos (partes)
    folder = "./data"
    parts = ["backlog.json"]
    parts = sorted(parts)

    TIEMPO_INICIAL = datetime.fromisoformat(tiempo_desde)
    TIEMPO_FINAL = datetime.fromisoformat(tiempo_hasta)

    def safe_load(path):
        with open(path, "r", encoding="utf-8") as f:
            txt = f.read()
        # por si hubiera coma colgante al cortar
        txt = re.sub(r",\s*([}\]])", r"\1", txt)
        return json.loads(txt)

    orders_list = []
    meta_cpt = None

    #preprocesamiento del backlog para pasarlo a dataframe
    for p in parts:
        data = safe_load(os.path.join(folder, p))
        # Algunos archivos podrían venir como {"metadata": {...}, "orders": [...]}
        if isinstance(data, dict) and "orders" in data:
            if meta_cpt is None and "metadata" in data and "cpt_hours" in data["metadata"]:
                meta_cpt = data["metadata"]["cpt_hours"]
            orders_list.extend(data["orders"])
        elif isinstance(data, list):
            orders_list.extend(data)
        else:
            if isinstance(data, dict):
                if "orders" in data:
                    orders_list.extend(data["orders"])
                else:
                    for v in data.values():
                        if isinstance(v, list) and v and isinstance(v[0], dict) and "order_id" in v[0]:
                            orders_list.extend(v)
                            break

    # A DataFrame
    df = pd.DataFrame(orders_list)

    # Convertir fechas a datetime
    df["creation_date"] = pd.to_datetime(df["creation_date"], errors="coerce")
    df["due_date"] = pd.to_datetime(df["due_date"], errors="coerce")

    # Filtrar órdenes que SÍ están en la ventana temporal
    # TIEMPO_INICIAL < creation_date <= TIEMPO_FINAL
    df_dentro_ventana = df[(df["creation_date"] > TIEMPO_INICIAL) & (df["creation_date"] <= TIEMPO_FINAL)].copy()

    df_dentro_ventana['m_vencimiento'] = (df['due_date'] - TIEMPO_FINAL).dt.total_seconds() / 60.0

    return df_dentro_ventana

## SS

In [6]:
def SS(N,df_dentro_ventana,estado_racks,total_no_procesadas_tm):
    #Mapeo inverso (obtener fila de dataframe)
    row_id_por_o = df_dentro_ventana.index.tolist()
    indices_no_procesadas_tm = [i for i in range(total_no_procesadas_tm)]

    #Procesamiento de las órdenes de la ventana para generar un rango finito en los números de órden y producto asociado:
    counter = 0
    tuplas_de_counter_item_id = []
    for index, row in df_dentro_ventana.iterrows():
        tuplas_de_counter_item_id.append((counter, row["item_id"]))
        counter += 1
    known_tuplas_de_counter_item_id = dict()
    tuplas_definitivas = []

    contador_numero_item = 0
    for tupla in tuplas_de_counter_item_id:
        item_id_siendo_procesado = tupla[1]
        if not item_id_siendo_procesado in known_tuplas_de_counter_item_id.keys():
            known_tuplas_de_counter_item_id[item_id_siendo_procesado] = contador_numero_item
            contador_numero_item += 1
        tuplas_definitivas.append((tupla[0], known_tuplas_de_counter_item_id[item_id_siendo_procesado]))

    p_por_o = np.array([p for (_, p) in tuplas_definitivas], dtype=int)

    #Para este punto, tuplas_definitivas guarda una lista de (o,p) con todos o's distintos
    O = len(tuplas_definitivas)

    with open("./data/stock.json", "r", encoding="utf-8") as f:
        stock_data = json.load(f)

    # Inicializar la lista de stock
    S_de_interes = []
    inverse_map_rack_id = {}
    cant_que_agrego = 0

    known_products_list = list(known_tuplas_de_counter_item_id.keys())

    # Procesar cada rack
    for rack_name, rack_data in sorted(stock_data.items()):
        rack_list = []

        # Definir todas las caras posibles (Cara_1 a Cara_4)
        caras_posibles = [f"Cara_{i}" for i in range(1, 5)]

        # Procesar cada cara posible
        for cara_name in caras_posibles:
            if cara_name in rack_data:
                productos = rack_data[cara_name]
                cara_list = []

                # Para cada producto, agregar el inventory_id según la cantidad
                for producto in productos:
                    inventory_id = producto["Inventory ID"]
                    cantidad = producto["Cantidad"]

                    # Agregar el inventory_id tantas veces como la cantidad
                    if inventory_id in known_products_list:
                        cara_list.append((known_tuplas_de_counter_item_id[inventory_id], cantidad))

                rack_list.append(cara_list)
            else:
                # Si la cara no existe en el JSON, agregar lista vacía
                rack_list.append([])
        if rack_list != [[],[],[],[]]:
          #Se genera para un rack la lista con 4 sublistas, una por cara, cada una con tuplas (p,cant) por cada p que haya en esa cara. Solo se genera si hay al menos
          #una cara no vacía
          r = len(S_de_interes)
          inverse_map_rack_id[r] = rack_name
          S_de_interes.append(rack_list)


    #Creación del modelo:
    problemTP = LpProblem("TP", LpMaximize)

    # Variables necesarias para el seteo del modelo
    O = len(tuplas_definitivas)
    C = 4
    R = len(S_de_interes)

    productos_numeros = [t[1] for t in tuplas_definitivas]
    P = (max(productos_numeros) + 1) if productos_numeros else 0

    # Renombramos por S, con esta variable chequearemos el stock disponible de cada producto
    S = S_de_interes

    #El N es el mínimo entre el que viene como input y el total de órdenes
    N = min(N,O)

    #Calculamos la cantidad de stock por producto en una cara de rack, para la primer restricción y necesario para crear la variable A:
    stock_rcp = {}
    productos_con_stock = set()
    for r in range(R):
        for c in range(len(S[r])):
            for (p, cant) in S[r][c]:
                if cant > 0:
                    stock_rcp[(r, c, p)] = stock_rcp.get((r, c, p), 0) + cant
                    productos_con_stock.add(p)

    #Estructuras para optimizar el modelo: generamos las combinaciones (orden,cara,rack,producto) factibles usando la estructura stock_rcp del paso anterior
    #para saber en una orden, en que caras de que rack tiene stock el producto asociado a la órden para que la variable de asignación del modelo solo se cree
    #si tiene sentido y podría valer 1 alguna vez
    indices_por_o = {o: [] for o in range(O)}
    indices_por_rcp = {}
    indices_por_rc = {}
    caras_rack_posibles = []
    indices = []
    for o in range(O):
        p = p_por_o[o]
        p_int = int(p)
        for (r,c,p2), cant in stock_rcp.items():

          if p2 != p:
              continue
          t = (o,c,r,p_int)
          if (c,r) not in caras_rack_posibles:
            caras_rack_posibles.append((c,r))
          indices.append(t)
          indices_por_o[o].append(t)
          indices_por_rc.setdefault((r,c), []).append(t)
          indices_por_rcp.setdefault((r,c,p), []).append(t)


    #Actualizacion de racks fríos y tibios (temperatura)
    #Recuperamos el estado de cada rack actualizado en la iteración anterior haciendo el mapeo inverso que nos da para un cierto r, su posición en
    #el array global que tiene a todos los racks, incluyendo los que no se consideran en esta iteración porque no tienen ningún producto de ninguna orden de la
    #ventana
    racks_frios = []
    racks_tibios = []
    for r in range(R):
      nombre_rack = inverse_map_rack_id[r]
      id_estado = int(nombre_rack.split("_")[1]) - 1
      if estado_racks[id_estado] == -3:
        racks_frios.append(r)
      else:
        racks_tibios.append(r)


    #Variables del modelo

    # A_o,c,r,p
    #Se asigna la orden o con producto p a la cara c del rack r. La optimización previa crea solo las variables "realizables"
    A = {t: LpVariable(f"A_{t[0]}_{t[1]}_{t[2]}_{t[3]}", 0, 1, cat="Binary") for t in indices}
    #B_c_r
    # Se usa la cara c del rack r
    B = LpVariable.dicts("B", (range(0, C), range(0, R)), lowBound=0, cat='Binary')
    #F_r
    # El rack frio es usado
    F = LpVariable.dicts("F", (range(0, R)), lowBound=0, cat='Binary')
    #T_r
    # El rack tibio es usado
    T = LpVariable.dicts("T", (range(0, R)), lowBound=0, cat='Binary')

    #RESTRICCIONES

    # 1) Nunca nos podemos exceder del stock disponible para una cara en determinado rack:
    # S_r,c,p >= ∑ A_o,c,r,p para todo c, r, p
    for r in range(0,R):
        for c in range(0,C):
            if (r,c) in indices_por_rc.keys():
              for t in indices_por_rc[r,c]:
                p = t[3]
                total_stock_producto = stock_rcp[(r,c,p)]
                problemTP += lpSum(A[t2] for t2 in indices_por_rcp[r,c,p]) <= total_stock_producto


    # 2) Para una orden en particular solo puede haber una selección:
    # ∑ ∑ ∑ A_o,c,r,p <= 1 para todo o
    for o in range(0,O):
        problemTP += lpSum(A[t] for t in indices_por_o[o]) <= 1

    # 3) No hay mas selecciones que el numero de ordenes total:
    # ∑ ∑ ∑ ∑ A_o,c,r,p <=  Cantidad de ordenes (es N)
    problemTP += lpSum(A[t] for t in indices) <= N

    # 4) Activacion de cara para determinado rack
    # de 0 hasta p ∑  A_o,c,r,p <= B_c,r para todo o,c,r
    for o in range(0,O):
      for t in indices_por_o[o]:
        c = t[1]
        r = t[2]
        problemTP += A[t] <= B[c][r]


    # Pickear el producto correcto para cada orden (restricción obsoleta pero que estuvo en la primer versión)
    # para cada o,c,r,p:
    # A_o,c,r,p <=  Delta_o,p
    #for o in range(O):
    #    for c in range(C):
    #        for r in range(R):
    #            for p in range(P):
    #                if (o,p) in delta_set_stock:
    #                    problemTP += A[o][c][r][p] <= 1

    # 5) Determinación de racks fríos. Un rack no puede estar frío en cierta iteración si alguna de sus caras no está activada
    for r in racks_frios:
        problemTP += lpSum(B[c][r] for c in range(0, 4)) <= 4 * F[r]
        problemTP += F[r] <= lpSum(B[c][r] for c in range(0, 4))

    # 6) Determinación de racks tibios. Un rack no puede estar tibio en cierta iteración si alguna de sus caras no está activada
    for r in racks_tibios:
        problemTP += lpSum(B[c][r] for c in range(0, 4)) <= 4 * T[r]
        problemTP += T[r] <= lpSum(B[c][r] for c in range(0, 4))

    #Calculamos la penalización por fecha de cada orden:
    F_o = np.zeros(len(df_dentro_ventana))
    for o in range(len(df_dentro_ventana)):
        minutos_vencimiento = df_dentro_ventana.iloc[o]["m_vencimiento"]
        F_o[o] = fec_o(minutos_vencimiento)

    # Función Objetivo (Ganancia)

    SLA_total = sum(F_o[o] for o in range(O))  #sumamos el costo de incumplir todas las que se vencen
    SLA_realizado = lpSum(F_o[o] * A[t] for o in range(O) for t in indices_por_o[o])
    SLA_real = SLA_total - SLA_realizado
    problemTP += (
        lpSum(8*2*A[t] for t in indices if t[0] in indices_no_procesadas_tm) #Maximizamos Asignaciones de ordenes no procesadas por el TM - priorizadas (suman más)
        + lpSum(4*2*A[t] for t in indices if t[0] not in indices_no_procesadas_tm) #Maximizamos Asignaciones del resto de órdenes de la ventana
        - lpSum(B[c][r] for c in range(C) for r in range(R)) #Penalizamos por cuantas más caras de rack usadas
        - lpSum(5 * F[r] for r in racks_frios) #Penalizamos 5 por cada rack frío usado
        - lpSum(3 * T[r] for r in racks_tibios) #Penalizamos 3 por cada rack frío usado
        - SLA_real #Penalizamos por cada orden no asignada tanto como corresponda según lo cerca a vencerse que está (cumplimiento SLA)
        )


    #Resolver el problema (llamado al solver)
    solver = pulp.PULP_CBC_CMD(
        msg=False,
        threads=4,          #cantidad de nodos del cpu a usar
        timeLimit= 120,     #límite de tiempo tolerable para buscar solución óptima
        gapRel=0.01         #gap relativo, es en qué porcentaje del óptimo cortar (por ejemplo si es 0,01 se para cuando se está en el 1% óptimo)
    )
    problemTP.solve(solver)

    #Recorrida de asignacioens para actualizar el estado de racks mapeando a los originales y saber los tibios y fríos en la siguiente iteración
    racks_frios_usados = 0
    racks_tibios_usados = 0
    ids_originales_rango = []
    for r in range(R):
      nombre_rack = inverse_map_rack_id[r]
      id_estado = int(nombre_rack.split("_")[1]) - 1
      ids_originales_rango.append(id_estado)
      cara_encendida = any( (value(B[c][r]) or 0.0) > 0 for c in range(C) ) #es 1 si hay alguna cara del rack que no sea 0
      if cara_encendida:
       if estado_racks[id_estado] == -3:
        racks_frios_usados+=1
       else:
        racks_tibios_usados+=1
       estado_racks[id_estado] = 0  #pasa a ser o se mantiene en tibio
      else:
          estado_racks[id_estado] = max(-3, estado_racks[id_estado] - 1) #o decrece su cantidad de usos o continua siendo frío

    total_racks_usados = racks_frios_usados + racks_tibios_usados
    porcentaje_frios_usados = (racks_frios_usados * 100 / total_racks_usados) if total_racks_usados > 0 else 0

    ids_no_usados = [i for i, valor in enumerate(estado_racks) if i not in ids_originales_rango]
    print("Racks tibios usados: ",racks_tibios_usados)
    print("Racks fríos usados: ",racks_frios_usados)
    for idx in ids_no_usados:
      estado_racks[idx] = max(-3, estado_racks[idx] - 1)

    ################ Script para actualizar el stock#########################

    vars_en_uno = [v.name for v in problemTP.variables() if v.varValue == 1]

    # Crear mapeo inverso de índice numérico a item_id
    inverse_map_product_id = {v: k for k, v in known_tuplas_de_counter_item_id.items()}

    # Lista para almacenar los item_id de los productos pickeados
    tuplas_productos_rack_cara_pickeadas = []

    # Procesar cada variable A activa
    for var_name in vars_en_uno:
        if var_name.startswith("A_"):
            # Extraer índices o, c, r, p del nombre de variable
            parts = var_name.split("_")
            if len(parts) == 5:
                o = int(parts[1])
                c = int(parts[2])
                r = int(parts[3])
                p = int(parts[4])
                # Convertir índice a item_id real
                tuple_product_rack_cara = inverse_map_product_id[p]
                rack_id = inverse_map_rack_id[r]
                tuplas_productos_rack_cara_pickeadas.append((tuple_product_rack_cara, rack_id, c))

    print(f"\nTotal de productos pickeados: {len(tuplas_productos_rack_cara_pickeadas)}")
    #print("\nProductos pickeados (formateado):")

    # 1. Cargar el stock actualizado
    with open("./data/stock.json", "r", encoding="utf-8") as f:
        stock_actualizado = json.load(f)

    # 2. Para cada producto pickeado, buscar y restar 1 unidad, eliminando la entrada completa si llega a 0
    print("Actualizando stock...")
    productos_eliminados = 0

    for tuple_product_rack_cara in tuplas_productos_rack_cara_pickeadas:
        encontrado = False
        # Buscar en todos los racks
        rack_data = stock_actualizado[tuple_product_rack_cara[1]]
        # Las caras en el modelo van de 0 a 3 y en el json de 1 a 4
        cara_name = f"Cara_{tuple_product_rack_cara[2] + 1}"
        if cara_name in rack_data:
            # Se busca el producto específico en esta cara
            for i, producto in enumerate(rack_data[cara_name][:]):  # Copia de la lista para iterar
                if producto["Inventory ID"] == tuple_product_rack_cara[0] and producto["Cantidad"] > 0:
                    producto["Cantidad"] -= 1
                    # Si la cantidad llega a 0, se elimina el registro
                    if producto["Cantidad"] == 0:
                        del rack_data[cara_name][i]
                        productos_eliminados += 1
                    encontrado = True
                    break

        if not encontrado:
            print(f"Advertencia: Producto {tuple_product_rack_cara[0]} no encontrado en stock! Error significativo, deberia haberlo encontrado ahi")

    # 3. Guardar el stock actualizado
    with open("./data/stock.json", "w", encoding="utf-8") as f:
      json.dump(stock_actualizado, f, indent=2, ensure_ascii=False)

    #print(f"Total de registros eliminados (cantidad 0): {productos_eliminados}")

    ###########################Fin actualización stock############################################


    #Devolvemos como output principal el pool de órdenes asignadas, agrupadas por cara de rack
    #También devolvemos la máxima cantidad de picks per face entre todas las caras de rack que tuvieron alguna asignación
    pool = defaultdict(list)
    ppf = 0
    for t in indices:
      key = ""
      if(value(A[t])) == 1:
        o = t[0]
        c = t[1]
        r = t[2]
        key = (f"Rack_{r+1}", f"Cara_{c+1}")
        pool[key].append(f"orden_{o+1}")
      if key != "":
        ppf = max(ppf,len(pool[key]))

    # Para guardar la lista de ordenes no procesadas en este pool por parte del SS para tener en el siguiente
    ordenes_no_procesadas = []
    vencimiento_minimo = 1000000000000000
    for o in range(O):
        vars_o = [A[t] for t in indices_por_o[o]]
        fue_asignada = sum((value(v) or 0) for v in vars_o)  #value es una funcion de pulp que devuelve el valor de la variable despues de resolver el modelo
        if fue_asignada ==0:
            ordenes_no_procesadas.append(o)
        else:
          #o asignada, usando que está asignada, guardamos para KPIs la orden asignada más cercana a vencerse
          row_id_ord = int(row_id_por_o[o])
          if df_dentro_ventana.loc[row_id_ord]["m_vencimiento"] < vencimiento_minimo:
            vencimiento_minimo = df_dentro_ventana.loc[row_id_ord]["m_vencimiento"]

    # Obtenemos las filas del dataset con las pendientes y las asignadas (disjuntos)
    ordenes_no_procesadas_row_ids = [int(row_id_por_o[o]) for o in ordenes_no_procesadas]
    pendientes_df = df_dentro_ventana.loc[ordenes_no_procesadas_row_ids]
    indices_pendientes = pendientes_df.index
    mascara_no_pendientes = ~df_dentro_ventana.index.isin(indices_pendientes)
    asignadas_df = df_dentro_ventana.loc[mascara_no_pendientes]
    asignadas = O - len(ordenes_no_procesadas)

    return pool, asignadas_df, pendientes_df, estado_racks, asignadas, ppf, vencimiento_minimo, porcentaje_frios_usados

## TM


In [10]:
import uuid

def simular_pendientes_tm(asignadas_df):
    """
    genera una cantidad aleatoria de ordenes no procesadas por el TM.
    """
    #Fija el total de órdenes que puede extraer en 100, si diera más que eso
    x = min(100,random.randint(0, len(asignadas_df)//10))
    df_pendientes_tm = asignadas_df.sample(x)
    return df_pendientes_tm

## WES


In [14]:
def WES():
    TIEMPO_DESDE = datetime.fromisoformat("2025-10-09 00:00:04") #Fecha de creación de la primer órden
    TIEMPO_HASTA = TIEMPO_DESDE + timedelta(minutes = 5) #Nos moveremos de a 5 minutos
    TIEMPO_MAX = datetime.fromisoformat("2025-10-09 23:59:59") #Fecha de creación de la última órden
    pendientes_df = pd.DataFrame()
    it = 0
    estado_racks = [-3] * 2089 #Sabemos que en stock.json el ID máximo de rack es 2089, este array global dirá quienes son fríos y tibios en todo momento
    sla_minimo = 1000000000000000 #Ponemos un valor muy grande así se va actualizando por más pequeños (KPI)
    ppfs_por_iteracion = [] #Guarda máx.ppf por iteración para después promediar (KPI)
    tiempos_pool = [] #Guardamos tiempo que tomó cada ciclo para promediar (KPI)
    frios_por_iteracion = [] #Guardamos cantidad de fríos a partir de la 2da iteración para ver promedio (KPI)
    no_procesadas_tm_ciclo_anterior = pd.DataFrame()
    while TIEMPO_HASTA <= TIEMPO_MAX:
        print("---------------------------------------------------------------------------")
        print("CICLO ",it+1)
        #Generamos la ventana de órdenes creadas en este intervalo
        ordenes_ventana = generar_ventana(TIEMPO_DESDE.isoformat(),TIEMPO_HASTA.isoformat())
        print("La ventana tiene ", len(ordenes_ventana), " órdenes")
        #Extraer las órdenes que ya tenemos confirmado que no se procesarán seguro en este ciclo de antemano (por el TM)
        no_procesadas_tm = simular_pendientes_tm(ordenes_ventana)
        indices_ventana = ordenes_ventana.index
        indices_no_procesadas = no_procesadas_tm.index
        print("El TM no procesará ",len(no_procesadas_tm), " órdenes")
        excluir_no_procesadas = ~ordenes_ventana.index.isin(indices_no_procesadas)
        ordenes_ciclo = ordenes_ventana.loc[excluir_no_procesadas]
        if not pendientes_df.empty:
            #Le restamos 5 minutos porque la órden viene de un pool anterior y ya le queda menos
            pendientes_df['m_vencimiento'] = np.maximum(0,pendientes_df['m_vencimiento'] - 5)
            ventana_df = pd.concat([pendientes_df, ordenes_ciclo])
        else:
            ventana_df = ordenes_ciclo

        if not no_procesadas_tm_ciclo_anterior.empty:
            #Le restamos 5 minutos como antes
            no_procesadas_tm_ciclo_anterior['m_vencimiento'] = np.maximum(0,no_procesadas_tm_ciclo_anterior['m_vencimiento'] - 20)
            ventana_df = pd.concat([no_procesadas_tm_ciclo_anterior, ventana_df])

        #Tomamos indices de las no procesadas por el TM del ciclo anterior para saber que órdenes son y priorizarlas
        indices_no_procesadas_ciclo_anterior = no_procesadas_tm_ciclo_anterior.index

        print(f"Ventana generada. Comenzando a correr pool entre horario {TIEMPO_DESDE} y {TIEMPO_HASTA}. Cantidad total de órdenes a procesar: {len(ordenes_ciclo)} además de {len(pendientes_df)} pendientes del ciclo anterior, y {len(no_procesadas_tm_ciclo_anterior)} no procesadas por el Task Manager")
        print("------------------------------------------------------------------")
        #Calculamos el tiempo de la ejecución para KPI
        start = time.time()
        N = random.randint(1500,2000) #Input, máx ordenes a procesar, cambia en cada pool
        print("Se procesarán como máximo ",N," órdenes en el pool")
        print("------------------------------------------------------------------")
        pool, asignadas_df, pendientes_df, estado_racks, asignadas, ppf, vencimiento_mas_cercano, frios_usados = SS(N,ventana_df, estado_racks,len(indices_no_procesadas_ciclo_anterior.tolist()))
        end = time.time()
        tiempo_pool_actual = end-start
        tiempos_pool.append(tiempo_pool_actual)
        if(it > 0):
          frios_por_iteracion.append(frios_usados)
        print(f"Pool obtenido")
        print(dict(pool))
        print("Maximos PPF en el pool: ",ppf)
        ppfs_por_iteracion.append(ppf)
        print("------------------------------------------------------------------")
        porcentaje_pendientes = round((100 * len(pendientes_df)) /(asignadas + len(pendientes_df)),2)
        print(f"Se han asignado {asignadas} ordenes y quedaron {len(pendientes_df)} ordenes pendientes (un {porcentaje_pendientes} %) ")
        print("------------------------------------------------------------------")
        print(f"Indices de las pendientes:{pendientes_df.index.tolist()}")
        print("------------------------------------------------------------------")
        if vencimiento_mas_cercano < sla_minimo:
          sla_minimo = vencimiento_mas_cercano
          print("Se actualiza el tiempo más cercano a vencerse por una orden asignada a ",sla_minimo, " minutos")

        TIEMPO_DESDE = TIEMPO_HASTA
        TIEMPO_HASTA = TIEMPO_DESDE +  timedelta(minutes = 5) #avanzamos 5 minutos todo para el siguiente ciclo
        it+=1
        no_procesadas_tm_ciclo_anterior = no_procesadas_tm

    #Para KPIs y output final
    ppf_promedio = np.median(ppfs_por_iteracion)
    tiempo_pool_promedio = np.median(tiempos_pool)
    frios_promedio = np.median(frios_por_iteracion)
    print("------------------------------------------------------------------")
    print("Métricas Finales de Evaluación de Performance")
    print("Eficiencia. Picks Per Face promedio a lo largo del SS: ",ppf_promedio)
    print("Productividad. Porcentaje de racks fríos usados en promedio a lo largo del SS: ", round(frios_promedio,2),"%")
    print("Cumplimiento del SLA. La orden asignada más cercana a vencerse fue a ", round(sla_minimo,2), " minutos de su due date")
    print("Calidad técnica del desarrollo. El tiempo promedio por obtención de un Pool fue de ", round(tiempo_pool_promedio,2), " segundos")

In [13]:
WES()

---------------------------------------------------------------------------
CICLO  1
La ventana tiene  327  órdenes
El TM no procesará  28  órdenes
Ventana generada. Comenzando a correr pool entre horario 2025-10-09 00:00:04 y 2025-10-09 00:01:04. Cantidad total de órdenes a procesar: 299 además de 0 pendientes del ciclo anterior, y 0 no procesadas por el Task Manager
------------------------------------------------------------------
Se procesarán como máximo  1640  órdenes en el pool
------------------------------------------------------------------
Racks tibios usados:  0
Racks fríos usados:  125

Total de productos pickeados: 237
Actualizando stock...
Pool obtenido
{('Rack_165', 'Cara_4'): ['orden_2'], ('Rack_33', 'Cara_3'): ['orden_3'], ('Rack_89', 'Cara_2'): ['orden_4'], ('Rack_116', 'Cara_1'): ['orden_8', 'orden_9', 'orden_299'], ('Rack_29', 'Cara_3'): ['orden_10', 'orden_11'], ('Rack_223', 'Cara_4'): ['orden_12', 'orden_13'], ('Rack_141', 'Cara_4'): ['orden_14', 'orden_15', 'ord